# Litmus: official MTEB AppsRetrieval run on Kaggle

Produces the screening file `appsretrieval_results.json` (plus the dense-only baseline).

**Before running**
1. Right panel → **Session options → Accelerator: GPU T4 x2** (or P100). **Internet: On** (needs a phone-verified Kaggle account).
2. Give the notebook the code, either way:
   * **Add Input → Upload** the file `litmus_source.zip` (from the project's `dist/` folder), or
   * put your public GitHub repo URL in `REPO_URL` below.
3. **Run All.** Expected time: roughly 45-75 min (GPU embeds ~12.5k texts; the sandbox executes candidates on the CPUs).
4. When finished, download the files from the **Output** panel (`/kaggle/working`).

The GPU only speeds up this one-off bulk embedding. The model and precision (fp32) are the same as on CPU, so the numbers match; the product itself runs on CPU.

In [ ]:
import os, glob, shutil, subprocess, zipfile
REPO_URL = ""  # e.g. "https://github.com/<you>/litmus-code-retrieval.git" - leave empty if you uploaded litmus_source.zip
WORK = "/kaggle/working/litmus"
if not os.path.exists(os.path.join(WORK, "pyproject.toml")):
    zips = glob.glob("/kaggle/input/**/litmus_source.zip", recursive=True)
    trees = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
    if zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/kaggle/working")          # the zip contains a top-level litmus/ folder
    elif trees:                                       # Kaggle auto-extracts uploaded zips
        shutil.copytree(trees[0], WORK, dirs_exist_ok=True)
    elif REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
    else:
        raise SystemExit("Upload litmus_source.zip as an input, or set REPO_URL")
os.chdir(WORK)
print(sorted(os.listdir(".")))

In [ ]:
!pip install -q -e ".[dev]" 2>&1 | tail -n 3
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# Self-test (includes the execution sandbox on Linux). Expect: 16 passed
!python -m pytest -q tests

In [ ]:
# 1) Dense-only baseline (the AbsEncoder from the guideline template). Also fills the embedding cache.
!LITMUS_HOME=/kaggle/working/.litmus LITMUS_DEVICE=cuda LITMUS_BATCH_SIZE=16 python scripts/run_mteb_eval.py --dense-only

In [ ]:
# 2) Full Litmus pipeline (dense recall + execution verification). Reuses the cached embeddings.
!LITMUS_HOME=/kaggle/working/.litmus LITMUS_DEVICE=cuda LITMUS_BATCH_SIZE=16 python scripts/run_mteb_eval.py --workers 4

In [ ]:
import json
for f in ["appsretrieval_results.json", "appsretrieval_results_dense.json", "appsretrieval_responses.csv"]:
    p = os.path.join(WORK, "results", f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join("/kaggle/working", f))
        if f.endswith(".json"):
            s = json.load(open(p))["scores"]["test"][0]
            print("%-36s NDCG@10 %.4f   MRR@10 %.4f" % (f, s["ndcg_at_10"], s["mrr_at_10"]))
print("Download these from the Output panel:", [f for f in os.listdir("/kaggle/working") if f.startswith("appsretrieval")])